<a href="https://colab.research.google.com/github/bhamboredisha/Traffic-Sign-Recognition-/blob/main/Trafficsign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
 # Download Dataset + Imports
# ============================================================

import kagglehub
path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print("Dataset downloaded to:", path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
from PIL import Image
from matplotlib.lines import Line2D
from tqdm.notebook import tqdm          # progress bar

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout
from sklearn.model_selection import train_test_split

print(f"✅ TensorFlow: {tf.__version__}")

# ============================================================
#  Paths + Load CSVs + Fix ALL Capitalisation
# ============================================================

dataset_dir = '/kaggle/input/gtsrb-german-traffic-sign'

metaDf  = pd.read_csv(os.path.join(dataset_dir, 'Meta.csv'))
trainDf = pd.read_csv(os.path.join(dataset_dir, 'Train.csv'))
testDf  = pd.read_csv(os.path.join(dataset_dir, 'Test.csv'))

# KEY FIX: CSV uses Train/ Test/ Meta/ but actual folders are lowercase
trainDf['Path'] = trainDf['Path'].str.replace('Train/', 'train/', regex=False)
testDf['Path']  = testDf['Path'].str.replace('Test/',  'test/',  regex=False)
metaDf['Path']  = metaDf['Path'].str.replace('Meta/',  'meta/',  regex=False)

# Build full absolute paths once — avoids repeated os.path.join in loops
trainDf['FullPath'] = dataset_dir + '/' + trainDf['Path']
testDf['FullPath']  = dataset_dir + '/' + testDf['Path']
metaDf['FullPath']  = dataset_dir + '/' + metaDf['Path']

print(f"✅ CSVs loaded — Train: {len(trainDf)} | Test: {len(testDf)} | Meta: {len(metaDf)}")

# Verify all three
for name, df in [('Train', trainDf), ('Test', testDf), ('Meta', metaDf)]:
    exists = os.path.exists(df['FullPath'].iloc[0])
    print(f"   {name}: {df['FullPath'].iloc[0]}")
    print(f"          Exists? {exists}")

labels = [
    '20 km/h', '30 km/h', '50 km/h', '60 km/h', '70 km/h',
    '80 km/h', '80 km/h end', '100 km/h', '120 km/h', 'No overtaking',
    'No overtaking for tracks', 'Crossroad with secondary way', 'Main road',
    'Give way', 'Stop', 'Road up', 'Road up for track', 'Brock',
    'Other dangerous', 'Turn left', 'Turn right', 'Winding road',
    'Hollow road', 'Slippery road', 'Narrowing road', 'Roadwork',
    'Traffic light', 'Pedestrian', 'Children', 'Bike', 'Snow', 'Deer',
    'End of the limits', 'Only right', 'Only left', 'Only straight',
    'Only straight and right', 'Only straight and left', 'Take right',
    'Take left', 'Circle crossroad', 'End of overtaking limit',
    'End of overtaking limit for track'
]

# ============================================================
# Fast Batch Image Loader
# ============================================================

IMG_SIZE   = 30
NUM_CLASSES = 43

def load_images_fast(paths, labels_list=None, size=IMG_SIZE):
    """
    Loads all images from a list of absolute paths using cv2.
    Shows a tqdm progress bar.
    Returns (X array, y array) or just X if labels_list is None.
    """
    images, ys, failed = [], [], 0

    for i, p in enumerate(tqdm(paths, desc="Loading images")):
        img = cv2.imread(p)
        if img is not None:
            img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (size, size))
            images.append(img)
            if labels_list is not None:
                ys.append(labels_list[i])
        else:
            # PIL fallback
            try:
                img = np.array(Image.open(p).convert("RGB").resize((size, size)))
                images.append(img)
                if labels_list is not None:
                    ys.append(labels_list[i])
            except:
                failed += 1

    print(f"  ✅ Loaded: {len(images)} | ❌ Failed: {failed}")
    X = np.array(images, dtype=np.float32) / 255.0
    if labels_list is not None:
        return X, np.array(ys, dtype=np.int32)
    return X

# Quick sanity check
_t = cv2.imread(trainDf['FullPath'].iloc[0])
print(f"✅ cv2 direct read: {'OK — shape ' + str(_t.shape) if _t is not None else '❌ FAILED'}")

# ============================================================
# Class Distribution Plots
# ============================================================

fig, axs = plt.subplots(1, 2, figsize=(25, 6))
sns.countplot(x="ClassId", hue="ClassId", data=trainDf,
              palette="Set1", legend=False, ax=axs[0])
sns.countplot(x="ClassId", hue="ClassId", data=testDf,
              palette="Set1", legend=False, ax=axs[1])
axs[0].set_title("Train Class Distribution", fontsize=14)
axs[1].set_title("Test Class Distribution",  fontsize=14)
plt.tight_layout()
plt.show()

# ============================================================
# KDE Plot (Width vs Height)
# ============================================================

trainSub = trainDf[(trainDf.Width < 80) & (trainDf.Height < 80)]
testSub  = testDf [(testDf.Width  < 80) & (testDf.Height  < 80)]

g = sns.JointGrid(x="Width", y="Height", data=trainSub, height=7)
sns.kdeplot(data=trainSub, x="Width", y="Height",
            cmap="Reds",  fill=False, ax=g.ax_joint)
sns.kdeplot(data=testSub,  x="Width", y="Height",
            cmap="Blues", fill=False, ax=g.ax_joint)
g.ax_joint.legend(handles=[
    Line2D([0], [0], color='red',  label='Train'),
    Line2D([0], [0], color='blue', label='Test')
])
g.ax_joint.set_title("KDE: Train (Red) vs Test (Blue)", pad=12)
plt.show()

# ============================================================
# Meta Images (one per class)
# ============================================================

metaDf = metaDf.sort_values(by=['ClassId']).reset_index(drop=True)

fig, axs = plt.subplots(6, 8, figsize=(25, 12))
idx = 0
for i in range(6):
    for j in range(8):
        ax = axs[i, j]
        ax.axis('off')
        if idx >= len(metaDf):
            break
        img      = cv2.imread(metaDf["FullPath"].iloc[idx])
        class_id = int(metaDf["ClassId"].iloc[idx])
        if img is not None:
            img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (60, 60))
            ax.imshow(img)
            ax.set_title(labels[class_id], fontsize=7)
        else:
            ax.imshow(np.ones((60, 60, 3), dtype=np.uint8) * 230)
            ax.set_title("N/A", fontsize=7, color="red")
        idx += 1

plt.suptitle("Meta Images — One Per Class", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ============================================================
#  Random 100 Training Images
# ============================================================

sampleDf    = trainDf.sample(n=100, random_state=42).reset_index(drop=True)
placeholder = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 230

fig, axs = plt.subplots(10, 10, figsize=(25, 25))
idx = 0
for i in range(10):
    for j in range(10):
        ax  = axs[i, j]
        row = sampleDf.iloc[idx]
        img = cv2.imread(row["FullPath"])
        if img is not None:
            img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
            ax.imshow(img)
            ax.set_title(labels[int(row["ClassId"])], fontsize=6)
        else:
            ax.imshow(placeholder)
            ax.set_title("Missing", fontsize=6, color="red")
        ax.axis('off')
        idx += 1

plt.suptitle("Random 100 Training Images", fontsize=14, y=1.005)
plt.tight_layout()
plt.show()

# ============================================================
#  Load Full Training Dataset (FAST)
# ============================================================

print("Loading full training dataset …")
X_train_full, y_train_full = load_images_fast(
    trainDf['FullPath'].tolist(),
    trainDf['ClassId'].tolist()
)

y_cat = to_categorical(y_train_full, NUM_CLASSES)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_cat,
    test_size=0.2, random_state=42, stratify=y_train_full
)
print(f"  X_train: {X_train.shape} | X_val: {X_val.shape}")
print("  ✅ Data ready!")

# ============================================================
#  Build & Train CNN Model
# ============================================================

model = Sequential([
    Conv2D(32, (5, 5), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (5, 5), activation='relu'),
    MaxPool2D(2, 2),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu'),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPool2D(2, 2),
    Dropout(0.25),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

history = model.fit(
    X_train, y_train,
    batch_size      = 64,
    epochs          = 5,
    validation_data = (X_val, y_val),
    verbose         = 1
)
print("✅ Training complete!")

# ============================================================
#  Training Curves
# ============================================================

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(history.history['accuracy'],     label='Train')
axs[0].plot(history.history['val_accuracy'], label='Val')
axs[0].set_title('Accuracy')
axs[0].set_xlabel('Epoch')
axs[0].legend()

axs[1].plot(history.history['loss'],     label='Train')
axs[1].plot(history.history['val_loss'], label='Val')
axs[1].set_title('Loss')
axs[1].set_xlabel('Epoch')
axs[1].legend()

plt.tight_layout()
plt.show()

# ============================================================
#  Load Test Dataset (FAST) + Evaluate
# ============================================================

print("Loading test images …")
X_test, y_test = load_images_fast(
    testDf['FullPath'].tolist(),
    testDf['ClassId'].tolist()
)

y_test_cat = to_categorical(y_test, NUM_CLASSES)

loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\n  ✅ Test Accuracy : {acc * 100:.2f}%")
print(f"     Test Loss     : {loss:.4f}")

# ============================================================
#   Visualise Test Predictions
# ============================================================

sample_idx  = np.random.choice(len(X_test), 25, replace=False)
predictions = model.predict(X_test[sample_idx])

fig, axs = plt.subplots(5, 5, figsize=(18, 18))
for i, ax in enumerate(axs.flat):
    true_label = labels[np.argmax(y_test_cat[sample_idx[i]])]
    pred_label = labels[np.argmax(predictions[i])]
    correct    = true_label == pred_label
    ax.imshow(X_test[sample_idx[i]])
    ax.set_title(f"T: {true_label}\nP: {pred_label}",
                 fontsize=7, color="green" if correct else "red")
    ax.axis('off')

plt.suptitle("Test Predictions — Green=Correct  Red=Wrong", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ============================================================
# Save Model to Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/Traffic_sign_Recognition/traffic_sign_model.keras'
model.save(save_path)
print(f"✅ Model saved to: {save_path}")

In [ ]:

import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))
# If it prints [] you are on CPU

In [ ]:
# Convert your notebook to a .py file
!jupyter nbconvert --to script /content/drive/MyDrive/Traffic_sign_Recognition/traffic_sign_recognition.ipynb \
    --output /content/drive/MyDrive/Traffic_sign_Recognition/traffic_sign_final
print("✅ Converted to .py")

In [ ]:
import os

# Check both possible save locations
for folder in [
    '/content/drive/MyDrive/Colab Notebooks',
    '/content/drive/MyDrive/Traffic_sign_Recognition'
]:
    if os.path.exists(folder):
        print(f"✅ {folder}")
        print("  ", os.listdir(folder))

In [ ]:
# Remove the sensitive commit history and start fresh
%cd /content/Traffic-Sign-Recognition-

!git rm --cached Trafficsign.ipynb
!git rm --cached traffic_sign_final_v3.py
!git rm --cached README.md

!git commit -m "Remove files to clean history"

In [ ]:
import json

notebook_path = '/content/drive/MyDrive/Colab Notebooks/Trafficsign.ipynb'

with open(notebook_path, 'r') as f:
    content = f.read()

# Remove any token patterns (ghp_ followed by any characters)
import re
content_clean = re.sub(r'ghp_[A-Za-z0-9]+', 'YOUR_TOKEN_HERE', content)

with open(notebook_path, 'w') as f:
    f.write(content_clean)

print("✅ All tokens removed from notebook!")

# Verify
if 'ghp_' in content_clean:
    print("⚠️ Some tokens still found!")
else:
    print("✅ Notebook is clean — no tokens found!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')